# MODEL_BASELINE — NextWatch

On compare une **baseline simple** a **3 modeles**. Tous proposent un top de films a partir d'un film aime (et du profil pour certains). Les films deja aimes en train sont exclus des suggestions.

## Lexique rapide des methodes

| Nom | En une phrase | Idee simple |
| --- | --- | --- |
| **Dummy** | Tire des films au hasard | Sert de plancher : si on ne fait pas mieux, le modele est inutile. |
| **Baseline** | Meme genre(s) que le film aime, classes par note moyenne | Regle du pouce sans ML : "dans le meme rayon, les mieux notes". |
| **KMeans (enrichi)** | Regroupe les films qui se ressemblent, puis propose dans le meme groupe | Ressemblance = **genres** + **profil du film** (note moyenne, popularite, public age/sexe sur le train). Puis on ajuste un peu selon l'age/sexe de l'utilisateur. |
| **Item-item CF** | "Ceux qui ont aime ce film ont aussi aime..." | Compare les films via les **notes des utilisateurs** (pas seulement les genres). |
| **LogisticRegression** | Estime la chance qu'un film plaise (note >= 4) | Apprend un score a partir de l'age, du sexe, des genres, etc., puis classe les candidats. |

**Metriques :** Precision@5 et Precision@10 (part de films vraiment aimes, note >= 4). **Seed :** 42.


## 1. Chargement et verification

Meme perimetre que l'EDA : exclusion `occupation` / `unknown`, genres en liste.


In [99]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# --- Hyperparametres ---
RANDOM_STATE = 42
# Reproductibilite (split, modeles, echantillons, dummy).

LIKED_THRESHOLD = 4
# Seuil a partir duquel une note compte comme "film aime" (cible y, requetes, pertinents).

MIN_SUPPORT = 10
# Nombre minimal de notes train pour utiliser la moyenne d'un film dans les scores.

TRAIN_FRAC = 0.7
# Proportion des notes reservee a l'apprentissage (le reste sert au test).

ALPHA_AGE = 0.35
# Amplitude du correctif age dans le score KMeans (parts de notes par genre selon la tranche).

ALPHA_SEX = 0.20
# Amplitude du correctif sexe dans le score KMeans (meme logique, poids separe et un peu plus faible).

CF_N_NEIGHBORS = 40
# Nombre de films voisins retenus par film seed pour le filtrage collaboratif item-item.

KMEANS_K_RANGE = range(12, 25)
# Valeurs de K explorees pour KMeans ; le K final est choisi par silhouette.

np.random.seed(RANDOM_STATE)

GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

DATA_DIR = next(
    (p for p in [Path("dataset"), Path("../dataset")] if (p / "u.data.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("dataset/ introuvable")

notes = pd.read_csv(DATA_DIR / "u.data.csv")
films = pd.read_csv(DATA_DIR / "u.item.csv")
users = pd.read_csv(DATA_DIR / "u.user.csv", dtype={"zip_code": "string"}).drop(columns=["occupation"])

df_raw = notes.merge(users, on="user_id", how="left").merge(
    films, left_on="item_id", right_on="movie_id", how="left"
)
print(f"Genre unknown exclu : {int((df_raw['unknown']==1).sum())} notes")

df = df_raw.loc[df_raw["unknown"] != 1].drop(columns=["unknown"]).copy()
df["genres"] = df[GENRES].apply(lambda row: [g for g in GENRES if row[g] == 1], axis=1)

print(f"df : {len(df):,} notes | {df['user_id'].nunique()} users | {df['item_id'].nunique()} films")
display(df[["user_id", "item_id", "rating", "age", "gender", "title", "genres"]].head(3))


Genre unknown exclu : 10 notes
df : 99,990 notes | 943 users | 1680 films


,user_id,item_id,rating,age,gender,title,genres
0,196,242,3,49,M,Kolya (1996),[Comedy]
1,186,302,3,39,F,L.A. Confidential (1997),"[Crime, Film-Noir, Mystery, Thriller]"
2,22,377,1,25,M,Heavyweights (1994),"[Children's, Comedy]"


## 2. Split sans fuite

Split aleatoire 80/20 des notes. Stats et fit modeles **uniquement sur le train**.


In [100]:
train = df.sample(frac=TRAIN_FRAC, random_state=RANDOM_STATE)
test = df.drop(train.index)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f"Train {len(train):,} | Test {len(test):,}")

movies = df.drop_duplicates("item_id").set_index("item_id")
movie_genres = movies["genres"].to_dict()
movie_title = movies["title"].to_dict()
# matrice genres 0/1 par film (catalogue)
film_genre_X = movies[GENRES].astype(float)
all_movie_ids = film_genre_X.index.to_numpy()
print(f"Films catalogue : {len(all_movie_ids)}")


Train 69,993 | Test 29,997
Films catalogue : 1680


## 3. Stats train + profil age/sexe

Note moyenne par film ; parts de notes par genre (re-rank KMeans).


In [101]:
film_stats = (
    train.groupby("item_id")["rating"].agg(n_ratings="count", mean_rating="mean").reset_index()
)
eligible = film_stats.loc[film_stats["n_ratings"] >= MIN_SUPPORT].set_index("item_id")
mean_rating = eligible["mean_rating"]
print(f"Films eligible (>= {MIN_SUPPORT} notes) : {len(eligible)}")

train = train.copy()
train["age_bin"] = pd.cut(
    train["age"], bins=[0, 17, 24, 34, 44, 54, 100],
    labels=["<=17", "18-24", "25-34", "35-44", "45-54", "55+"],
)

def preference_share(frame, group_col):
    out = {}
    for key, sub in frame.groupby(group_col, observed=True):
        lists = sub["genres"]
        out[key] = {g: lists.map(lambda gs, genre=g: genre in gs).mean() for g in GENRES}
    return pd.DataFrame(out)

pref_age = preference_share(train, "age_bin")
pref_gender = preference_share(train, "gender")
global_share = {g: train["genres"].map(lambda gs, genre=g: genre in gs).mean() for g in GENRES}
age_boost = pref_age.sub(pd.Series(global_share), axis=0)
gender_boost = pref_gender.sub(pd.Series(global_share), axis=0)
display(pref_age.loc[["Drama", "Action"]].round(3))


Films eligible (>= 10 notes) : 1056


,<=17,18-24,25-34,35-44,45-54,55+
Drama,0.334,0.363,0.386,0.411,0.470,0.471
Action,0.301,0.287,0.268,0.244,0.223,0.192


## 4. Cas d'evaluation

Requete A = meilleur film aime en train ; pertinents = aimes en test.
On conserve aussi la liste des films **deja aimes en train** pour les exclure des tops (evite de "reussir" en reproposant le passe).


In [102]:
liked_train = train.loc[train["rating"] >= LIKED_THRESHOLD]
liked_test = test.loc[test["rating"] >= LIKED_THRESHOLD]
train_counts = train.groupby("item_id").size()
cand_A = liked_train.copy()
cand_A["pop"] = cand_A["item_id"].map(train_counts)
idx_A = (
    cand_A.sort_values(["rating", "pop"], ascending=[False, False])
    .groupby("user_id", sort=False).first().reset_index()
)[["user_id", "item_id", "rating", "age", "gender"]]
idx_A = idx_A.rename(columns={"item_id": "query_id", "rating": "query_rating"})

liked_lists = liked_train.groupby("user_id")["item_id"].apply(set)
relevant = liked_test.groupby("user_id")["item_id"].apply(set).rename("relevant_ids").reset_index()
eval_cases = idx_A.merge(relevant, on="user_id", how="inner")
eval_cases["liked_train_ids"] = eval_cases["user_id"].map(liked_lists)
eval_cases["relevant_ids"] = eval_cases.apply(
    lambda r: r["relevant_ids"] - {r["query_id"]}, axis=1
)
eval_cases = eval_cases.loc[eval_cases["relevant_ids"].map(len) > 0].reset_index(drop=True)
eval_cases["age_bin"] = pd.cut(
    eval_cases["age"], bins=[0, 17, 24, 34, 44, 54, 100],
    labels=["<=17", "18-24", "25-34", "35-44", "45-54", "55+"],
)
print(f"Cas d'evaluation : {len(eval_cases)}")
print(f"Taille moyenne pertinents : {eval_cases['relevant_ids'].map(len).mean():.1f}")
display(eval_cases.head(3))


Cas d'evaluation : 939
Taille moyenne pertinents : 17.8


,user_id,query_id,query_rating,age,gender,relevant_ids,liked_train_ids,age_bin
0,419,50,5,37,M,"{1, 514, 100, 617, 1451, 173, 14, 604}","{257, 134, 269, 275, 286, 300, 174, 50, 306, 1...",35-44
1,549,50,5,42,M,"{1, 515, 748}","{288, 866, 258, 100, 237, 50, 405, 118, 181, 1...",35-44
2,296,50,5,43,F,"{898, 1284, 7, 1160, 137, 654, 14, 272, 274, 1...","{1, 514, 515, 521, 9, 11, 523, 528, 20, 22, 23...",35-44


## 5. Entrainement des modeles (train uniquement)

### KMeans enrichi — regrouper les films
On ne clusterise plus seulement avec les genres. Chaque film devient un vecteur :
- ses **genres** (0/1) ;
- sa **note moyenne** et son **nombre de notes** (train) ;
- le **profil de son public** : part femmes / hommes et parts par tranches d'age parmi ceux qui l'ont note (train).

Ces chiffres sont **standardises** (meme echelle) puis on lance KMeans ; le meilleur nombre de groupes K est choisi avec la silhouette.

### Item-item CF — films souvent aimes ensemble
On place chaque film dans l'espace des notes utilisateurs, puis on trouve ses voisins (cosine).

### LogisticRegression — proba d'aimer
On apprend sur des exemples train : le film B est-il aime (note >= 4) selon le profil et le contenu.


In [103]:
# --- KMeans enrichi : genres + profil notes du film (train only) ---
tr = train.copy()
tr["age_bin"] = pd.cut(
    tr["age"], bins=[0, 17, 24, 34, 44, 54, 100],
    labels=["<=17", "18-24", "25-34", "35-44", "45-54", "55+"],
)
AGE_BINS = ["<=17", "18-24", "25-34", "35-44", "45-54", "55+"]

film_n = train.groupby("item_id").size().rename("n_ratings")
film_mean = train.groupby("item_id")["rating"].mean().rename("mean_rating_raw")
film_female = train.assign(is_F=train["gender"].eq("F")).groupby("item_id")["is_F"].mean().rename("share_F")
age_share = (
    tr.groupby(["item_id", "age_bin"], observed=True).size()
    .unstack(fill_value=0)
    .reindex(columns=AGE_BINS, fill_value=0)
)
age_share = age_share.div(age_share.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
age_share.columns = [f"share_age_{c}" for c in age_share.columns]

feat = film_genre_X.copy()
feat["mean_rating_raw"] = film_mean.reindex(feat.index)
feat["log_n_ratings"] = np.log1p(film_n.reindex(feat.index).fillna(0))
feat["share_F"] = film_female.reindex(feat.index)
for c in age_share.columns:
    feat[c] = age_share[c].reindex(feat.index)

global_mean = float(train["rating"].mean())
feat["mean_rating_raw"] = feat["mean_rating_raw"].fillna(global_mean)
feat["share_F"] = feat["share_F"].fillna(0.5)
for c in age_share.columns:
    feat[c] = feat[c].fillna(1.0 / len(AGE_BINS))

cluster_feature_cols = list(feat.columns)
film_cluster_X = feat[cluster_feature_cols].astype(float)

train_film_ids = train["item_id"].unique()
X_fit_raw = film_cluster_X.loc[film_cluster_X.index.isin(train_film_ids)]
cluster_scaler = StandardScaler()
X_fit = cluster_scaler.fit_transform(X_fit_raw.values)
X_all = cluster_scaler.transform(film_cluster_X.values)

sil_scores = {}
for k in KMEANS_K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_fit)
    if len(set(labels)) < 2:
        continue
    sil_scores[k] = silhouette_score(X_fit, labels)
best_k = max(sil_scores, key=sil_scores.get)
print("Silhouette (clustering enrichi):", {k: round(v, 3) for k, v in sil_scores.items()})
print("K choisi:", best_k)
print(f"Features clustering ({len(cluster_feature_cols)}) : genres + note + log(n) + share_F + parts age")

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_fit)
film_cluster = pd.Series(kmeans.predict(X_all), index=film_cluster_X.index, name="cluster")
print("Taille des clusters (extrait) :")
print(film_cluster.value_counts().sort_index().head())


# --- Item-item CF (notes) ---
users_u = sorted(train["user_id"].unique())
items_u = sorted(train["item_id"].unique())
u_ix = {u: i for i, u in enumerate(users_u)}
i_ix = {it: i for i, it in enumerate(items_u)}
mat = csr_matrix(
    (train["rating"].astype(float).values,
     (train["item_id"].map(i_ix).values, train["user_id"].map(u_ix).values)),
    shape=(len(items_u), len(users_u)),
)
nn_cf = NearestNeighbors(metric="cosine", algorithm="brute")
nn_cf.fit(mat)
cf_item_ids = np.array(items_u)
cf_index = {mid: i for i, mid in enumerate(items_u)}
print(f"CF item-item : {mat.shape[0]} films x {mat.shape[1]} users")

# Precompute CF neighbor lists (sim) for speed
CF_NEI = {}
n_ask = min(CF_N_NEIGHBORS + 1, len(items_u))
dists_all, idxs_all = nn_cf.kneighbors(mat, n_neighbors=n_ask)
for i, mid in enumerate(items_u):
    pairs = []
    for d, j in zip(dists_all[i], idxs_all[i]):
        other = int(cf_item_ids[j])
        if other == mid:
            continue
        pairs.append((other, 1.0 - float(d)))
    CF_NEI[mid] = pairs
print(f"Voisins CF precomputes pour {len(CF_NEI)} films")

# --- LogisticRegression ---
sample = train.sample(n=min(30000, len(train)), random_state=RANDOM_STATE)
users_liked = liked_train.groupby("user_id")["item_id"].apply(list)
rng = np.random.default_rng(RANDOM_STATE)
fit_rows = []
for _, row in sample.iterrows():
    uid, bid = int(row["user_id"]), int(row["item_id"])
    liked = users_liked.get(uid, [])
    others = [x for x in liked if x != bid]
    aid = int(rng.choice(others)) if others else bid
    ga, gb = set(movie_genres.get(aid, [])), set(movie_genres.get(bid, []))
    if bid not in mean_rating.index:
        continue
    feat = {
        "age": row["age"],
        "gender_M": 1 if row["gender"] == "M" else 0,
        "overlap": len(ga & gb),
        "mean_rating": float(mean_rating.loc[bid]),
        "same_cluster": int(film_cluster.get(aid) == film_cluster.get(bid)),
        "y": int(row["rating"] >= LIKED_THRESHOLD),
    }
    for g in GENRES:
        feat[f"b_{g}"] = int(g in gb)
    fit_rows.append(feat)

logit_df = pd.DataFrame(fit_rows)
feature_cols = [c for c in logit_df.columns if c != "y"]
scaler = StandardScaler()
X_log = scaler.fit_transform(logit_df[feature_cols])
y_log = logit_df["y"].values
logit = LogisticRegression(max_iter=800, random_state=RANDOM_STATE, class_weight="balanced")
logit.fit(X_log, y_log)
print(f"LogReg : {len(logit_df):,} paires | y=1 rate={y_log.mean():.3f}")


Silhouette (clustering enrichi): {12: 0.166, 13: 0.177, 14: 0.18, 15: 0.184, 16: 0.203, 17: 0.193, 18: 0.2, 19: 0.218, 20: 0.219, 21: 0.226, 22: 0.23, 23: 0.229, 24: 0.234}
K choisi: 24
Features clustering (27) : genres + note + log(n) + share_F + parts age
Taille des clusters (extrait) :
cluster
0     78
1    276
2    223
3     49
4     50
Name: count, dtype: int64
CF item-item : 1621 films x 943 users
Voisins CF precomputes pour 1621 films
LogReg : 29,068 paires | y=1 rate=0.561


## 6. Comment on recommande (en pratique)

Pour chaque methode : partir du film aime A (et du profil si besoin), construire un score, garder le top-k.
**Regle commune :** on n'propose ni A, ni les films deja aimes par l'utilisateur en train.


In [104]:
def _seen(query_id, liked_train_ids):
    s = set(liked_train_ids) if liked_train_ids is not None else set()
    s.add(int(query_id))
    return s


def recommend_dummy(query_id, k=10, age=None, gender=None, age_bin=None, liked_train_ids=None):
    seen = _seen(query_id, liked_train_ids)
    pool = np.array([m for m in all_movie_ids if m not in seen])
    local = np.random.default_rng(RANDOM_STATE + int(query_id))
    if len(pool) == 0:
        return []
    return local.choice(pool, size=min(k, len(pool)), replace=False).tolist()


def recommend_baseline(query_id, k=10, age=None, gender=None, age_bin=None, liked_train_ids=None):
    seen = _seen(query_id, liked_train_ids)
    gq = set(movie_genres.get(query_id, []))
    if not gq:
        return []
    scores = [
        (mid, float(m))
        for mid, m in mean_rating.items()
        if mid not in seen and gq & set(movie_genres.get(mid, []))
    ]
    scores.sort(key=lambda x: (-x[1], x[0]))
    return [m for m, _ in scores[:k]]


def profile_genre_boost(age_bin, gender):
    b_age = pd.Series(0.0, index=GENRES)
    b_sex = pd.Series(0.0, index=GENRES)
    if age_bin in age_boost.columns:
        b_age = age_boost[age_bin].fillna(0)
    if gender in gender_boost.columns:
        b_sex = gender_boost[gender].fillna(0)
    return b_age, b_sex


def recommend_kmeans(query_id, k=10, age=None, gender=None, age_bin=None, liked_train_ids=None):
    seen = _seen(query_id, liked_train_ids)
    if query_id not in film_cluster.index:
        return []
    cid = film_cluster.loc[query_id]
    members = film_cluster[film_cluster == cid].index
    b_age, b_sex = profile_genre_boost(age_bin, gender)
    scores = []
    for mid in members:
        if mid in seen or mid not in mean_rating.index:
            continue
        base = float(mean_rating.loc[mid])
        gs = movie_genres.get(mid, [])
        if gs:
            pa = float(np.mean([b_age.get(g, 0.0) for g in gs]))
            ps = float(np.mean([b_sex.get(g, 0.0) for g in gs]))
        else:
            pa = ps = 0.0
        scores.append((mid, base + ALPHA_AGE * pa + ALPHA_SEX * ps))
    scores.sort(key=lambda x: (-x[1], x[0]))
    return [m for m, _ in scores[:k]]


def recommend_cf(query_id, k=10, age=None, gender=None, age_bin=None, liked_train_ids=None, user_id=None):
    # Multi-seed : tous les films aimes train, ponderes par leur note ; voisins CF precomputes
    seen = _seen(query_id, liked_train_ids)
    seeds = liked_train_ids if liked_train_ids is not None else {query_id}
    # notes des seeds
    seed_ratings = {}
    if user_id is not None:
        sub = train.loc[
            (train["user_id"] == user_id) & (train["rating"] >= LIKED_THRESHOLD),
            ["item_id", "rating"],
        ]
        seed_ratings = dict(zip(sub["item_id"], sub["rating"]))
    scores = {}
    for aid in seeds:
        if aid not in CF_NEI:
            continue
        w = float(seed_ratings.get(aid, LIKED_THRESHOLD))
        for mid, sim in CF_NEI[aid]:
            if mid in seen:
                continue
            scores[mid] = scores.get(mid, 0.0) + w * sim
    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


def recommend_logit(query_id, k=10, age=None, gender=None, age_bin=None, liked_train_ids=None):
    seen = _seen(query_id, liked_train_ids)
    ga = set(movie_genres.get(query_id, []))
    rows, ids = [], []
    for mid in mean_rating.index:
        if mid in seen:
            continue
        gb = set(movie_genres.get(mid, []))
        feat = {
            "age": float(age) if age is not None else 30.0,
            "gender_M": 1 if gender == "M" else 0,
            "overlap": len(ga & gb),
            "mean_rating": float(mean_rating.loc[mid]),
            "same_cluster": int(film_cluster.get(query_id) == film_cluster.get(mid)),
        }
        for g in GENRES:
            feat[f"b_{g}"] = int(g in gb)
        rows.append(feat)
        ids.append(mid)
    if not rows:
        return []
    X = scaler.transform(pd.DataFrame(rows)[feature_cols])
    proba = logit.predict_proba(X)[:, 1]
    order = np.argsort(-proba)
    return [ids[i] for i in order[:k]]



ex = eval_cases.iloc[0]
qid = int(ex["query_id"])
base_kw = dict(
    age=ex["age"], gender=ex["gender"], age_bin=ex["age_bin"],
    liked_train_ids=ex["liked_train_ids"],
)
print("Query:", movie_title.get(qid), "| cluster", int(film_cluster.get(qid)))
print("Dummy   :", [movie_title.get(i) for i in recommend_dummy(qid, 5, **base_kw)])
print("Baseline:", [movie_title.get(i) for i in recommend_baseline(qid, 5, **base_kw)])
print("KMeans  :", [movie_title.get(i) for i in recommend_kmeans(qid, 5, **base_kw)])
print("CF      :", [movie_title.get(i) for i in recommend_cf(qid, 5, **base_kw, user_id=int(ex["user_id"]))])
print("LogReg  :", [movie_title.get(i) for i in recommend_logit(qid, 5, **base_kw)])


Query: Star Wars (1977) | cluster 6
Dummy   : ['To Gillian on Her 37th Birthday (1996)', '8 Heads in a Duffel Bag (1997)', 'Ninotchka (1939)', 'Ace Ventura: Pet Detective (1994)', 'Philadelphia (1993)']
Baseline: ['Casablanca (1942)', "Schindler's List (1993)", 'Affair to Remember, An (1957)', 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)', 'Princess Bride, The (1987)']
KMeans  : ['Casablanca (1942)', "Schindler's List (1993)", 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)', 'Bridge on the River Kwai, The (1957)', 'Empire Strikes Back, The (1980)']
CF      : ['Silence of the Lambs, The (1991)', 'Empire Strikes Back, The (1980)', 'E.T. the Extra-Terrestrial (1982)', 'Back to the Future (1985)', 'Wizard of Oz, The (1939)']
LogReg  : ['Casablanca (1942)', "Schindler's List (1993)", 'Close Shave, A (1995)', 'Affair to Remember, An (1957)', 'Shawshank Redemption, The (1994)']


## 7. Evaluation Precision@5 et @10


In [105]:
def precision_at_k(recommended, relevant, k):
    top = recommended[:k]
    if not top:
        return 0.0
    return sum(1 for m in top if m in relevant) / k


def evaluate(recommend_fn, cases, ks=(5, 10), max_k=10, pass_user_id=False):
    rows, cache = [], []
    for _, row in cases.iterrows():
        qid = int(row["query_id"])
        relevant = row["relevant_ids"]
        kwargs = dict(
            age=row["age"],
            gender=row["gender"],
            age_bin=row["age_bin"],
            liked_train_ids=row["liked_train_ids"],
        )
        if pass_user_id:
            kwargs["user_id"] = int(row["user_id"])
        rec = recommend_fn(qid, k=max_k, **kwargs)
        cache.append({"user_id": row["user_id"], "query_id": qid, "rec": rec, "relevant": relevant})
        rows.append({f"P@{k}": precision_at_k(rec, relevant, k) for k in ks})
    return pd.DataFrame(rows).mean().to_frame("mean").T, cache


methods = [
    ("dummy_random", recommend_dummy, False),
    ("baseline_overlap_mean", recommend_baseline, False),
    ("model_kmeans", recommend_kmeans, False),
    ("model_item_cf", recommend_cf, True),
    ("model_logistic", recommend_logit, False),
]

summaries, caches = [], {}
for name, fn, need_uid in methods:
    s, c = evaluate(fn, eval_cases, pass_user_id=need_uid)
    summaries.append(s.assign(method=name))
    caches[name] = c

results = pd.concat(summaries, ignore_index=True)[["method", "P@5", "P@10"]]
print(f"Precision moyenne sur {len(eval_cases)} cas (seed={RANDOM_STATE})")
display(results.round(4))


Precision moyenne sur 939 cas (seed=42)


,method,P@5,P@10
0,dummy_random,0.0079,0.0098
1,baseline_overlap_mean,0.1078,0.0912
2,model_kmeans,0.0931,0.0736
3,model_item_cf,0.2777,0.2319
4,model_logistic,0.0771,0.0742


## 8. Test situation produit (âge + sexe + un film)

Même métriques **P@5 / P@10**, mais entrée comme sur l'app finale : seulement **âge**, **sexe** et **un film aimé**.

- Pas de `user_id`, pas d'historique MovieLens.
- On n'exclut que le film saisi (pas les autres films aimés en train).
- Le CF ne garde qu'**un seed** (= le film requête) : c'est la version réaliste produit, plus faible que le multi-seed de la section 7.


In [106]:
def evaluate_product(recommend_fn, cases, ks=(5, 10), max_k=10):
    """Eval produit : entrée = âge, sexe, film seul (pas d'historique user)."""
    rows = []
    for _, row in cases.iterrows():
        qid = int(row["query_id"])
        relevant = row["relevant_ids"]
        # Situation app : un seul film connu → seed = {qid}, exclusion = {qid}
        rec = recommend_fn(
            qid,
            k=max_k,
            age=row["age"],
            gender=row["gender"],
            age_bin=row["age_bin"],
            liked_train_ids={qid},
        )
        rows.append({f"P@{k}": precision_at_k(rec, relevant, k) for k in ks})
    return pd.DataFrame(rows).mean().to_frame("mean").T


methods_product = [
    ("dummy_random", recommend_dummy),
    ("baseline_overlap_mean", recommend_baseline),
    ("model_kmeans", recommend_kmeans),
    ("model_item_cf", recommend_cf),       # 1 seed = film saisi
    ("model_logistic", recommend_logit),
]

summaries_prod = []
for name, fn in methods_product:
    s = evaluate_product(fn, eval_cases)
    summaries_prod.append(s.assign(method=name))

results_product = pd.concat(summaries_prod, ignore_index=True)[["method", "P@5", "P@10"]]
print(
    f"Situation produit (âge + sexe + 1 film) — Precision moyenne "
    f"sur {len(eval_cases)} cas (seed={RANDOM_STATE})"
)
display(results_product.round(4))

# Rappel section 7 (avec historique) pour comparer le CF
print("\nRappel section 7 (éval avec historique train, CF multi-seed) :")
display(results.round(4))


Situation produit (âge + sexe + 1 film) — Precision moyenne sur 939 cas (seed=42)


,method,P@5,P@10
0,dummy_random,0.0106,0.0096
1,baseline_overlap_mean,0.0754,0.0705
2,model_kmeans,0.0720,0.0585
3,model_item_cf,0.1427,0.1200
4,model_logistic,0.0592,0.0496



Rappel section 7 (éval avec historique train, CF multi-seed) :


,method,P@5,P@10
0,dummy_random,0.0079,0.0098
1,baseline_overlap_mean,0.1078,0.0912
2,model_kmeans,0.0931,0.0736
3,model_item_cf,0.2777,0.2319
4,model_logistic,0.0771,0.0742


## 9. Analyse d'erreurs (baseline)

Trois cas ou la baseline a 0 hit@5.


In [107]:
cache_base = caches["baseline_overlap_mean"]
error_rows = [c for c in cache_base if not any(m in c["relevant"] for m in c["rec"][:5])]
print(f"Baseline 0 hit@5 : {len(error_rows)} / {len(cache_base)}")
error_rows = sorted(error_rows, key=lambda c: -len(c["relevant"]))[:3]

for i, c in enumerate(error_rows, 1):
    qid = c["query_id"]
    print("=" * 70)
    print(f"Erreur #{i} — user {c['user_id']}")
    print(f"  A : {movie_title.get(qid)} | genres={movie_genres.get(qid)} | cluster={film_cluster.get(qid)}")
    print("  Top5 baseline :")
    for mid in c["rec"][:5]:
        print(f"    - {movie_title.get(mid)} | mean={mean_rating.get(mid, float('nan')):.2f} | cl={film_cluster.get(mid)}")
    print(f"  Pertinents test (n={len(c['relevant'])}, echantillon) :")
    for mid in list(c["relevant"])[:5]:
        print(f"    - {movie_title.get(mid)} | genres={movie_genres.get(mid)} | cl={film_cluster.get(mid)}")


Baseline 0 hit@5 : 601 / 939
Erreur #1 — user 758
  A : Fargo (1996) | genres=['Crime', 'Drama', 'Thriller'] | cluster=22
  Top5 baseline :
    - Close Shave, A (1995) | mean=4.48 | cl=5
    - Schindler's List (1993) | mean=4.47 | cl=6
    - Third Man, The (1949) | mean=4.41 | cl=4
    - Raise the Red Lantern (1991) | mean=4.28 | cl=1
    - Some Folks Call It a Sling Blade (1993) | mean=4.27 | cl=15
  Pertinents test (n=80, echantillon) :
    - Men in Black (1997) | genres=['Action', 'Adventure', 'Comedy', 'Sci-Fi'] | cl=7
    - Contact (1997) | genres=['Drama', 'Sci-Fi'] | cl=7
    - My Left Foot (1989) | genres=['Drama'] | cl=1
    - Quest, The (1996) | genres=['Action', 'Adventure'] | cl=13
    - Dumb & Dumber (1994) | genres=['Comedy'] | cl=2
Erreur #2 — user 130
  A : Star Wars (1977) | genres=['Action', 'Adventure', 'Romance', 'Sci-Fi', 'War'] | cluster=6
  Top5 baseline :
    - Casablanca (1942) | mean=4.47 | cl=6
    - Schindler's List (1993) | mean=4.47 | cl=6
    - Affair to 